In this notebook I will take in my basefile (with strike treatment and covariates added)

create a train/test split 

and explore some meta learners

In [1]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

from econml.metalearners import SLearner, TLearner, XLearner
from econml.dml import CausalForestDML

e:\tfl_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Here I bring in my basefile which is saved as a parquet.

This basefile contains London TFL bike station usage data per station per hour over a three period. This is my target or Y. This data is from tfl free to download data

Joined onto this basefile is a selection of covariates, of which you would expect to have some impact on the the bike usage each hour. This is my (observed confounders) covariate data or X. This is from time based data and weather sources.

Next we have a column with if that station at that hour had a line serve it that a strike on that day. This is my treatment column or T.

I am interested in the causal inference question: What is the effect of strike days on the usage of TFL bikes?

I calculate the CATE (conditional average treatment effect) using several META learner methods.

In [2]:
PATH = "bike_hourly_with_covariates_v2_filtered.parquet"  # adjust

df = pd.read_parquet(PATH)

df.size
df.columns

Index(['station_id', 'trips_start', 'date', 'ts', 'y_log1p', 'strike_exposed',
       'lat', 'lon', 'station_name', 'weather_cell_id', 'hour', 'dow', 'month',
       'year', 'doy', 'is_weekend', 'is_am_peak', 'is_pm_peak',
       'is_bank_holiday', 'is_school_holiday', 'temperature_2m',
       'relative_humidity_2m', 'precipitation', 'rain', 'cloud_cover',
       'wind_speed_10m', 'weather_code', 'dist_nearest_tube_km',
       'n_tube_within_500m', 'n_tube_within_1km', 'strike_severity_daily_frac',
       'days_to_next_strike', 'days_since_last_strike', 'cycle_infra_score'],
      dtype='str')

In [3]:
df.head()


,station_id,trips_start,date,ts,y_log1p,strike_exposed,lat,lon,station_name,weather_cell_id,...,cloud_cover,wind_speed_10m,weather_code,dist_nearest_tube_km,n_tube_within_500m,n_tube_within_1km,strike_severity_daily_frac,days_to_next_strike,days_since_last_strike,cycle_infra_score
0,39,2016-08-18 10:00:00,2016-08-18,3,1.386294,0,51.526377,-0.07813,"Shoreditch High Street, Shoreditch",g1.100_lat5214_lon-5,...,94.0,8.3,3.0,0.339435,1,3,0.0,30.0,30.0,188
1,39,2016-08-18 11:00:00,2016-08-18,4,1.609438,0,51.526377,-0.07813,"Shoreditch High Street, Shoreditch",g1.100_lat5214_lon-5,...,92.0,10.4,3.0,0.339435,1,3,0.0,30.0,30.0,188
2,39,2016-08-18 12:00:00,2016-08-18,7,2.079442,0,51.526377,-0.07813,"Shoreditch High Street, Shoreditch",g1.100_lat5214_lon-5,...,74.0,10.6,2.0,0.339435,1,3,0.0,30.0,30.0,188
3,39,2016-08-18 13:00:00,2016-08-18,6,1.945910,0,51.526377,-0.07813,"Shoreditch High Street, Shoreditch",g1.100_lat5214_lon-5,...,66.0,9.4,2.0,0.339435,1,3,0.0,30.0,30.0,188
4,39,2016-08-18 14:00:00,2016-08-18,1,0.693147,0,51.526377,-0.07813,"Shoreditch High Street, Shoreditch",g1.100_lat5214_lon-5,...,46.0,11.5,1.0,0.339435,1,3,0.0,30.0,30.0,188


In [4]:
COL_TIME = "trips_start"
COL_Y    = "ts"
COL_T    = "strike_exposed"

df[COL_TIME] = pd.to_datetime(df[COL_TIME])
df[COL_Y] = pd.to_numeric(df[COL_Y], errors="coerce")
df[COL_T] = pd.to_numeric(df[COL_T], errors="coerce").fillna(0).astype(int)

df = df.dropna(subset=[COL_TIME, COL_Y])
df["y_log1p"] = np.log1p(df[COL_Y].astype(float))

df[[COL_TIME, COL_Y, COL_T, "y_log1p"]].head()

,trips_start,ts,strike_exposed,y_log1p
0,2016-08-18 10:00:00,3,0,1.386294
1,2016-08-18 11:00:00,4,0,1.609438
2,2016-08-18 12:00:00,7,0,2.079442
3,2016-08-18 13:00:00,6,0,1.945910
4,2016-08-18 14:00:00,1,0,0.693147


We apply basic feature transformations and use a time based modelling split

In [5]:
SPLIT_DATE = "2018-01-01"

train = df[df[COL_TIME] < SPLIT_DATE].copy()
test  = df[df[COL_TIME] >= SPLIT_DATE].copy()

print("train:", train.shape, "test:", test.shape)

Y_train = train["y_log1p"].values
T_train = train[COL_T].values.astype(int)

Y_test = test["y_log1p"].values
T_test = test[COL_T].values.astype(int)

train: (5192715, 34) test: (2651643, 34)


In [6]:
# Candidate categorical features
#cat_cols = ["station_id"]  # add "weather_cell_id" if you want

# Candidate numeric features (only keep those that exist)
num_cols = [
    "hour", 
    "dow",
    "month",
    "year",
    "doy",
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "rain",
    "cloud_cover",
    "wind_speed_10m",
    "is_weekend",
    "is_am_peak",
    "is_pm_peak",
    "is_bank_holiday",
    "lat",
    "lon",
    'is_school_holiday',
    'dist_nearest_tube_km',
    'n_tube_within_500m', 
    'n_tube_within_1km', 
    'strike_severity_daily_frac',
    'days_to_next_strike',
    'days_since_last_strike',
    'cycle_infra_score'
]


#cat_cols = [c for c in cat_cols if c in train.columns]
num_cols = [c for c in num_cols if c in train.columns]

#print("cat_cols:", cat_cols)
print("num_cols:", num_cols)

pre = ColumnTransformer(
    transformers=[
      #  ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols),
    ],
    remainder="drop",
    sparse_threshold=0.3
)

Xtr = pre.fit_transform(train[num_cols])
Xte = pre.transform(test[num_cols])

print("Xtr:", Xtr.shape, "Xte:", Xte.shape)

num_cols: ['hour', 'dow', 'month', 'year', 'doy', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'rain', 'cloud_cover', 'wind_speed_10m', 'is_weekend', 'is_am_peak', 'is_pm_peak', 'is_bank_holiday', 'lat', 'lon', 'is_school_holiday', 'dist_nearest_tube_km', 'n_tube_within_500m', 'n_tube_within_1km', 'strike_severity_daily_frac', 'days_to_next_strike', 'days_since_last_strike', 'cycle_infra_score']
Xtr: (5192715, 25) Xte: (2651643, 25)


In [7]:
print("Treatment rate:", T_train.mean())
print("Treated count:", T_train.sum(), "of", len(T_train))

Treatment rate: 0.0057162775157119155
Treated count: 29683 of 5192715


Instantiate two Random Forest items - Regressor and Classifier

In [8]:
rf_y = RandomForestRegressor(
    n_estimators=200,
    min_samples_leaf=50,
    n_jobs=-1,
    random_state=0,
)

rf_t = RandomForestClassifier(
    n_estimators=200,
    min_samples_leaf=50,
    n_jobs=-1,
    random_state=0,
)

Finally I look at training meta learners using my basefile and random forest objects

We train a S,T and X learner.

These all show different levels of CATE, with the T learner actually showing a negative effect.

In [9]:
import time

start = time.time()

s = SLearner(overall_model=rf_y)
s.fit(Y_train, T_train, X=Xtr)
tau_s = s.effect(Xte)

print("SLearner done in", time.time() - start/ 60, "minutes")

Slearner_end = time.time()

t = TLearner(models=RandomForestRegressor(
    n_estimators=200, min_samples_leaf=100, n_jobs=-1, random_state=0
))
t.fit(Y_train, T_train, X=Xtr)
tau_t = t.effect(Xte)

print("TLearner done in", time.time() - Slearner_end / 60, "minutes")

Tlearner_end = time.time()

x = XLearner(models=RandomForestRegressor(
    n_estimators=200, min_samples_leaf=100, n_jobs=-1, random_state=0
))
x.fit(Y_train, T_train, X=Xtr)
tau_x = x.effect(Xte)

print("XLearner done in", time.time() - Tlearner_end/ 60, "minutes")

print("ATE S:", float(np.mean(tau_s)))
print("ATE T:", float(np.mean(tau_t)))
print("ATE X:", float(np.mean(tau_x)))

e:\tfl_project\.venv\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


SLearner done in 1743547965.6731608 minutes


e:\tfl_project\.venv\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


TLearner done in 1743551602.849051 minutes


e:\tfl_project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
e:\tfl_project\.venv\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


XLearner done in 1743559329.3283792 minutes
ATE S: 6.491867555191316e-05
ATE T: -0.07094002847863107
ATE X: 0.012294695971612505


In [10]:
from econml.dml import CausalForestDML

cf = CausalForestDML(
    model_y=RandomForestRegressor(n_estimators=200, min_samples_leaf=50, n_jobs=-1, random_state=0),
    model_t=RandomForestClassifier(n_estimators=200, min_samples_leaf=50, n_jobs=-1, random_state=0),
    n_estimators=200,
    min_samples_leaf=50,
    n_jobs=-1,
    random_state=0
)
cf.fit(Y_train, T_train, X=Xtr)
tau_cf = cf.effect(Xte)
lb, ub = cf.effect_interval(Xte, alpha=0.05)  # 95% confidence intervals
print("ATE CF:", np.mean(tau_cf))
print("ATE 95% CI:", np.mean(lb), "to", np.mean(ub))

AttributeError: Cannot use a classifier as a first stage model when the target is continuous!

We are currently violating some of the assumptions needed for causal inference /meta learners (?)

SUTVA is being violated. SUTVA is made up of no interference and no hidden versions of treatment. Both are being violated.

no interference: We need that the potential outcome for unit i depends only on unit i's own treatment, not on the treatment assigned to any other unit.

For our situation, a strike at one tube line can affect other tube stations near it, even if those other stations have a strike exposed value of 0.

Remedy - aggregate up station hour basefile to area-hour.

No hidden version of treatment: requires that there is one and only one version of T=1. We have differing levels of treatment, for example a single tube line striking on one day is not equivalent to 3 tube lines all striking on the same day.

Constitency: The observed outcome for a unit that received treatment t equals the potential outcome under treatment t. Strikes being announces on a particular day most probably align with conditions that make workers want to strike - wage disputes, political environments, inflations

Remedy - redefine treatment as a continuous variable instead of binary.

